# 第9回 体験ワーク① ── あなたの「パケットの旅」をたどる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

千葉大学 情報リテラシ（工学部1年）。第8回（ネットワーク回）でやり残した**手を動かす演習**です。
スライドの「URLを入れてからページが出るまでの旅」を、自分のコードで確かめます。
**読むだけでなく、上から順にセルを実行**してください（セルを選んで **Shift+Enter**）。

> 用語メモ：**パケット**＝ネットを流れるデータを小さく区切った「荷物」。
> **LAN**＝家や学校の中など、同じ建物・同じWi-Fiで結ばれた「ご近所ネットワーク（町内）」。

## このノートでやる5つのワーク
1. **ワーク1：IPアドレス＝ネット上の「住所」** ── 32ビットを2進数で見る／自分のグローバルIPを調べる
2. **ワーク2：DNS ＝ ネットの「電話帳」** ── 名前（example.com）から住所（IP）を引く
3. **ワーク3：サブネット ＝ 「同じ町内か？」の判定** ── `/24` に置ける台数と、宛先が同じLANかの判定
4. **ワーク4：パケットの中身をのぞく** ── `IP:443` に直接たずねて、流れる荷物（バイト列）を見る
5. **ワーク5：HTTPの会話を読む** ── メソッド・ステータス・ヘッダ・Cookie を自分の目で確かめる

> 🌐 と書いたセルは**インターネット接続が必要**です（Colab では最初からつながっています）。

---
## ワーク1：IPアドレス＝ネット上の「住所」（32ビット）

IPv4アドレスは **32ビット**。8ビット（＝1バイト）ずつ4つに区切り、各かたまりを10進数 **0〜255** で書きます（例：`192.168.1.10`）。
255 までなのは、8ビットで表せる数が \(2^8 = 256\) 個（0〜255）だからです。

各ビットには右から `1, 2, 4, 8, 16, 32, 64, 128` の「重み」があります。
たとえば `192` は `11000000` ＝ `128 + 64`。まず1かたまり（0〜255）を変換して、重みを目で見てみましょう。


In [ ]:
def to_bin8(n):
    """0〜255 の整数を 8ビットの2進文字列にする"""
    assert 0 <= n <= 255, "各オクテットは 0〜255 の範囲"
    return format(n, "08b")

def show_weights(n):
    bits = to_bin8(n)
    weights = [128, 64, 32, 16, 8, 4, 2, 1]
    print(f"{n} を2進数にすると: {bits}")
    print("重み  :", "  ".join(f"{w:>3}" for w in weights))
    print("ビット:", "  ".join(f"{b:>3}" for b in bits))
    used = [w for w, b in zip(weights, bits) if b == "1"]
    print("足し算:", " + ".join(map(str, used)) if used else "0", "=", n)

for v in [192, 168, 1, 10, 255]:
    show_weights(v)
    print("-" * 42)

# 👇 ここを好きな数（0〜255）に変えて実行してみよう
show_weights(100)

### IPアドレス全体（4かたまり）を2進数で見る
`192.168.1.10` のように4つ並べると、全部で `8 × 4 = 32` ビットになります。


In [ ]:
def ip_to_bin(ip):
    octets = [int(x) for x in ip.split(".")]
    assert len(octets) == 4 and all(0 <= o <= 255 for o in octets), "不正なIPv4です"
    return ".".join(to_bin8(o) for o in octets)

for ip in ["192.168.1.10", "10.0.0.1", "172.16.5.4"]:
    print(f"{ip:>15}  ->  {ip_to_bin(ip)}   (全{len(ip_to_bin(ip).replace('.',''))}ビット)")

### 🌐 自分の「グローバルIPアドレス」を調べる
スマホやPCがインターネットに出るときの“表向きの住所”です。
（同じWi-Fiの人どうしは同じグローバルIPに見えることが多い ＝ **NAT** のしくみ。スライド「NATとIPの節約」を思い出そう）


In [ ]:
# 🌐 要ネット：外部サービスに「私のIPは？」と尋ねる
import urllib.request, json

def my_global_ip():
    url = "https://api.ipify.org?format=json"
    with urllib.request.urlopen(url, timeout=10) as r:
        return json.load(r)["ip"]

try:
    ip = my_global_ip()
    print("あなたのグローバルIPアドレス:", ip)
    print("2進数で見ると            :", ip_to_bin(ip) if ip.count(".") == 3 else "(IPv6 のようです)")
except Exception as e:
    print("取得できませんでした（ネット未接続かも）:", e)

---
## ワーク2：DNS ＝ インターネットの「電話帳」

人は `www.chiba-u.ac.jp` のような**名前**を覚えますが、機械は**IPアドレス（数字の住所）**で通信します。
この「名前 → 住所」の変換をしてくれるのが **DNS** です。電話帳で名前から番号を引くのと同じ。


In [ ]:
# 🌐 要ネット：名前からIPアドレスを引く（DNS）
import socket

sites = ["www.chiba-u.ac.jp", "www.google.com", "example.com"]
for host in sites:
    try:
        ip = socket.gethostbyname(host)
        print(f"{host:>22}  ->  {ip}")
    except Exception as e:
        print(f"{host:>22}  ->  解決できませんでした ({e})")

### 1つの名前に、住所が複数あることもある
大きなサイトは世界中にサーバを置いて負荷を分散します（**CDN**）。
だから同じ名前でも、複数のIPが返ってくることがあります。


In [ ]:
# 🌐 要ネット：1つの名前に結びつく複数のIPを見る
import socket

host = "www.google.com"   # 👈 好きなサイト名に変えてOK
infos = socket.getaddrinfo(host, 443, proto=socket.IPPROTO_TCP)
ips = sorted({info[4][0] for info in infos})
print(f"{host} に結びつくIP（{len(ips)}件）:")
for ip in ips:
    print("  -", ip)

### 🌐 `dig` で「電話帳の生の答え」をのぞく（プロのツール）
ネットワーク技術者が実際に使うコマンドが **`dig`** です。
名前を引いたとき、DNSが返してくる「住所(A)・有効期限(TTL)・どのサーバが答えたか」まで丸ごと見られます。
Colab には最初は入っていないので、1行目で導入してから使います（数十秒かかることがあります）。


In [ ]:
# 🌐 要ネット：dig コマンドを使えるようにする（dnsutils を導入）
!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y dnsutils > /dev/null 2>&1
!dig -v 2>&1 | head -1 || echo "dig が見つかりません"

In [ ]:
# 🌐 要ネット：dig で名前解決の中身を見る
# +noall +answer … 答え（Answer）の部分だけを表示する見やすいオプション
!echo "===== www.chiba-u.ac.jp の A レコード =====" && dig www.chiba-u.ac.jp +noall +answer
!echo "" && echo "===== www.google.com（住所だけ手早く） =====" && dig www.google.com A +short
!echo "" && echo "===== example.com（左から: 名前 / TTL秒 / クラス(IN) / 種別(A) / 住所） =====" && dig example.com +noall +answer

> 表の見方（形の例）：`example.com.  3600  IN  A  203.0.113.10`
> ＝左から「名前 ／ TTL（秒）／ クラス（`IN`＝インターネット）／ 種別（`A`＝住所レコード）／ 住所（IP）」。
> つまり「example.com の住所はこのIP。この答えは 3600秒（＝1時間）覚えておいてよい（**TTL**＝有効期限）」という意味。
> TTL があるおかげで、毎回 電話帳（DNS）に問い合わせず、**手元に一時メモ（キャッシュ＝覚えておくこと）**して高速化できます。
>
> ※ 住所（IP）の数字は時期やサーバ運用で変わります。上のセルで実際に返ってきた値を見てください（`203.0.113.10` は説明用のダミーです）。


---
## ワーク3：サブネット ＝ 「同じ町内か？」の判定

端末（スマホ・PC）は荷物（パケット＝ネットを流れるデータのかたまり）を送るとき、
宛先が**同じLAN（町内）にいるか／外（よその町）か**を毎回判定します。
その境界を決めるのが **サブネットマスク**（`/24` などの数字。「住所の何ビットまでが町名か」を表す目印）です。

`192.168.1.0/24` は「上位24ビットが町名、残り8ビットが番地」という意味。
番地は \(2^8 = 256\) 通りありますが、うち2つは予約されています。

- **ネットワーク番号**（番地が全部0）… 「町そのもの」を指す住所
- **ブロードキャスト**（番地が全部1）… 「町内全員へ一斉送信」のための住所

この2つは機器に割り当てられないので、**置ける機器は最大 254 台（＝256 − 2）**です。


In [ ]:
import ipaddress

net = ipaddress.ip_network("192.168.1.0/24")
print("ネットワーク     :", net)
print("ネットワーク番号 :", net.network_address, "（予約：町そのものを指す）")
print("ブロードキャスト :", net.broadcast_address, "（予約：町内全員へ）")
print("サブネットマスク :", net.netmask)
print("置ける機器の台数 :", net.num_addresses - 2, "台")
print()
print("使えるアドレスの最初の5個:")
for i, host in enumerate(net.hosts()):
    if i >= 5:
        break
    print("  -", host)

### 宛先は「同じLAN」？それとも「外」？ ── マスクで“町名”を取り出して比べる

端末は通信のたびに、宛先が **同じ町内（同じLAN）か／よその町（外）か** を判定しています。
やり方はシンプル：**自分のIPと相手のIPの両方を「マスク」と AND して、残った『町名（ネットワーク部）』が一致するか**を見るだけ。

- **AND** ＝ 両方のビットが `1` のときだけ `1`、それ以外は `0`。
- マスク `/24` は上位24ビットが `1`。だから AND すると **下位8ビット（番地）が消えて、上位24ビット（町名）だけ残る**。
- 2つのIPの「町名」が同じ → 同じLAN（直接とどける）。違う → 外（ルータに渡す）。

下のセルは、その **AND を1ビットずつ目で見えるように** 2進数で並べて判定します。


In [ ]:
import ipaddress

def to_bits(ip):
    """'192.168.1.10' -> '11000000.10101000.00000001.00001010'（目で見る用）"""
    return ".".join(format(int(o), "08b") for o in ip.split("."))

def mask_dotted(prefix):
    """/24 -> '255.255.255.0' のマスク（10進）"""
    bits = "1" * prefix + "0" * (32 - prefix)
    return ".".join(str(int(bits[i:i+8], 2)) for i in range(0, 32, 8))

def network_part(ip, prefix):
    """IP と マスクを AND して『町名（ネットワーク部）』だけ取り出す"""
    ip_int   = int(ipaddress.ip_address(ip))
    mask_int = (0xFFFFFFFF << (32 - prefix)) & 0xFFFFFFFF   # 上位prefixビットだけ1
    return ipaddress.ip_address(ip_int & mask_int)          # AND で番地(下位)を0にする

def same_lan_step(ip_a, ip_b, prefix=24):
    mask = mask_dotted(prefix)
    print(f"=== {ip_a} から {ip_b} へ送りたい  (マスク /{prefix}) ===")
    print(f"マスク /{prefix} = {mask}")
    print(f"          ビット: {to_bits(mask)}")
    print(f"  （1の桁＝町名 / 0の桁＝番地。ANDすると番地が消えて『町名』だけ残る）\n")

    towns = []
    for ip in (ip_a, ip_b):
        town = network_part(ip, prefix)
        towns.append(town)
        print(f"  住所  {ip:>15} = {to_bits(ip)}")
        print(f"  AND   {'マスク':>15} = {to_bits(mask)}")
        print(f"  ──────────────────  ↓ AND（両方1の桁だけ残る）")
        print(f"  町名  {str(town):>15} = {to_bits(str(town))}\n")

    if towns[0] == towns[1]:
        print(f"判定：町名が一致（{towns[0]}）→ 🏠 同じLAN内！ 相手に直接とどける")
    else:
        print(f"判定：町名が違う（{towns[0]} ≠ {towns[1]}）→ 🚪 外。ルータ(出口)に渡す")
    print("=" * 64)

pairs = [
    ("192.168.1.10", "192.168.1.200"),   # 同じ /24（同じ町内）
    ("192.168.1.10", "192.168.2.5"),     # 隣の町（3つ目が違う）
    ("192.168.1.10", "8.8.8.8"),         # 完全に外（インターネット）
]
for a, b in pairs:
    same_lan_step(a, b)

---
## ワーク4：パケットの中身をのぞく ── 「住所:ポート番号」に直接たずねる

ふだんはブラウザのURLバーに `https://example.com` と打つだけ。
でもその裏でブラウザは、**IP(建物の住所) の 443番ポート(部屋番号=HTTPS)** あてに荷物（パケット）を届けています。

> スライドの言葉：`192.0.2.1:443` ＝「この住所の **443号室（HTTPS）** へ届けて」。

このワークでは **URLバーを使わず、自分で `IP:443` に接続**して、
実際に**流れる荷物の中身（バイト列）＝パケット**を目で見ます。


In [ ]:
# 🌐 要ネット：まず宛先を「住所:部屋番号」の形にする
import socket

host = "example.com"          # 👈 行き先のサイト名
ip = socket.gethostbyname(host)   # DNSで 名前→住所(IP)
port = 443                     # HTTPS の部屋番号

print(f"行き先のサイト名     : {host}")
print(f"宛先の建物(IPアドレス): {ip}")
print(f"宛先の部屋(ポート)    : {port}   (= HTTPS)")
print(f"────────────────────────────")
print(f"つまり荷物のあて先     : {ip}:{port}")

### 荷物は「入れ子」に包まれて旅する（カプセル化）

宛先 `IP:443` が決まったら、いよいよ荷物を送り出します。このとき荷物は **一度に作られるのではなく、上の層から下の層へ渡るたびに、各層が自分の「宛名シール（ヘッダ）」を前に貼って包んで**いきます。これが **カプセル化**。

- 中身（ほしいページ）→ アプリ層 → 輸送層が「部屋番号」シール → ネット層が「住所」シール → リンク層が「次の機器」シール
- 下に行くほど **外側のシールが増えて、荷物は長くなる**。
- 受け取った側は **外側のシールから1枚ずつはがして**、中身にたどり着く（＝逆カプセル化）。

下のセルは、実際に文字列を継ぎ足して **荷物が育っていく → はがされて元に戻る** ようすを1ステップずつ表示します。


In [ ]:
# カプセル化 ＝ 下の層へ行くたび「宛名シール(ヘッダ)」を前に貼って包む。
# 実際に文字列を継ぎ足して、荷物が“育っていく”ようすを見る。
import socket

host = "example.com"
try:
    ip = socket.gethostbyname(host)   # ワーク4で引いた住所(IP)
except Exception:
    ip = "93.184.216.34"              # 取れなければ説明用のサンプル
port = 443

# 各層が足す「宛名シール(ヘッダ)」を、上の層から順に並べる
steps = [
    ("アプリ層(HTTP)", f"GET / Host:{host}",      "ほしいページの中身そのもの"),
    ("輸送層(TCP)",    f"[TCP 宛先ポート:{port}]",  "どの部屋へ＋順番・再送の管理"),
    ("ネット層(IP)",   f"[IP 宛先:{ip}]",          "どの建物（住所）へ"),
    ("リンク層",       f"[MAC 次の機器へ]",         "目の前の次の1台へ"),
]

print("【送信側】上の層 → 下の層へ。シールを前に足して“包んで”いく\n")
packet = ""
for name, header, role in steps:
    packet = (header + " " + packet) if packet else header   # ★前に足す＝外側に包む
    print(f"▼ {name} が「{header}」を貼る … {role}")
    print(f"   いまの荷物 : {packet}")
    print(f"   荷物の長さ : {len(packet)} 文字\n")

print(f"→ これが実際にケーブル/電波を流れる荷物。一番外側＝{steps[-1][0]}のシールから読まれる。\n")
print("=" * 64)
print("\n【受信側】届いたら、外側のシールから1枚ずつ“はがして”中身を取り出す\n")

peel = packet
for name, header, role in reversed(steps):
    if name.startswith("アプリ層"):
        print(f"▲ シールを全部はがし終わり → 中身に到達！")
        print(f"   取り出した中身 : {peel}")
        break
    print(f"▲ {name} のシール「{header}」をはがす")
    peel = peel.replace(header + " ", "", 1)
    print(f"   残り       : {peel}\n")

### 🌐 `IP:443` へ TLS で接続し、手書きのリクエストを送ってみる
同じ建物（IP）に複数のサイトが同居していることがあるので、
封筒の表に「**どのサイト宛か（Host / SNI）**」も書きます。それさえ書けば、住所と部屋番号だけで本物のページに届きます。


In [ ]:
# 🌐 要ネット：IP:443 に直接つないで、流れるパケット(バイト列)を見る
import socket, ssl

host = "example.com"
ip = socket.gethostbyname(host)
port = 443

# 1) 送る「荷物の中身」を自分で組み立てる（これがHTTPパケットの本文）
request = (
    f"GET / HTTP/1.1\r\n"
    f"Host: {host}\r\n"                 # 建物の中で「どのサイト宛か」
    f"User-Agent: chiba-infolit\r\n"
    f"Connection: close\r\n"
    f"\r\n"
)
print("===== これから送る荷物（リクエスト・パケットの中身） =====")
print(request)

# 2) IP:443 へ「封筒に鍵をかけて(TLS)」接続する
ctx = ssl.create_default_context()
raw = socket.create_connection((ip, port), timeout=10)
tls = ctx.wrap_socket(raw, server_hostname=host)   # server_hostname=SNI（どのサイト宛か）
print(f"接続成功 → {ip}:{port}   暗号方式: {tls.version()}")

# 3) 送って、返ってきた荷物(レスポンス)を受け取る
tls.sendall(request.encode())
data = b""
while len(data) < 2000:
    chunk = tls.recv(4096)
    if not chunk:
        break
    data += chunk
tls.close()

# 4) 返ってきた荷物の「宛名シール(ヘッダ)」と中身の先頭を見る
text = data.decode("utf-8", "ignore")
head, _, body = text.partition("\r\n\r\n")
print("\n===== 返ってきた荷物のヘッダ（宛名シール） =====")
print(head)
print("\n===== 中身(HTML)の先頭 =====")
print(body[:300].strip())

> **いま何が起きた？**
> - URLバーを一切使わず、`IP:443`（住所:部屋番号）だけで本物のWebページに到達した。
> - 送った荷物も返ってきた荷物も、正体は**ただの文字列（バイト列）** ── これがパケットの中身。
> - `443`(HTTPS) なので、途中の機器にはこの文字列が**暗号化されて**しか見えない（だから野良Wi-Fiでも安全）。
>   もし `80`(HTTP, 平文) なら、この文字列が通り道に**丸見え**になります。次のノート②で確かめます。


---
## ワーク5：HTTPの会話を読む（メソッド・ステータス・ヘッダ・Cookie）

ワーク4で `IP:443` に荷物を届けたら、相手から返事が来ましたね。
この**やり取りの作法**が **HTTP** です。基本はとてもシンプルな1往復：

> 🗣️ こちら「**GET**（このページ**ください**）」 → 🖥️ サーバ「**200 OK**（はい、どうぞ）」

スライドの CHAPTER 2「Webを見る ― HTTPとWWW」で出てきた4つのキーワードを、コードで読み解きます。

| 部品 | 正体 | 例 |
|---|---|---|
| **メソッド** | サーバへの「**動詞**」 | `GET`（もらう）・`POST`（送る） |
| **ステータス** | 返事の**3桁の番号** | `200`成功 / `301`引越し / `403`立入禁止 / `404`見つからない / `500`サーバ故障 |
| **Content-Type** | 中身の**種類** | `text/html`（HTML）・`image/png`（画像） |
| **Set-Cookie** | 状態を覚える**合言葉** | `Set-Cookie: id=abc123`（次回から「さっきの人」と分かる） |

HTTPは**毎回相手を忘れる（ステートレス）**ので、ログイン状態などは Cookie で覚えさせます。
では、本物のサーバとの「会話」を盗み聞きしてみましょう。

In [ ]:
# 🌐 要ネット：HTTPの会話（リクエストとレスポンス）を関数にする
import socket, ssl

def http_get(host, path="/"):
    """host の 443番ポートへ TLS で GET し、(ステータス行, ヘッダdict, 本文先頭) を返す。"""
    # 1) 送る「動詞」＝ GET。封筒の表に Host（どのサイト宛か）も書く
    request = (
        f"GET {path} HTTP/1.1\r\n"
        f"Host: {host}\r\n"
        f"User-Agent: chiba-infolit\r\n"
        f"Connection: close\r\n"   # 受け取ったら接続を閉じてね、の合図
        f"\r\n"
    )
    # 2) IP:443 へ TLS（鍵つき封筒）で接続して送る
    ctx = ssl.create_default_context()
    raw = socket.create_connection((host, 443), timeout=10)
    tls = ctx.wrap_socket(raw, server_hostname=host)   # SNI＝どのサイト宛か
    tls.sendall(request.encode())

    # 3) 返事（レスポンス）を受け取る
    data = b""
    while True:
        chunk = tls.recv(4096)
        if not chunk:
            break
        data += chunk
        if len(data) > 8000:   # 先頭だけで十分なので打ち切る
            break
    tls.close()

    # 4) 「ヘッダ（宛名シール）」と「本文」に切り分ける
    text = data.decode("utf-8", "ignore")
    head, _, body = text.partition("\r\n\r\n")
    header_lines = head.split("\r\n")
    status_line = header_lines[0]            # 例: HTTP/1.1 200 OK
    headers = {}
    for line in header_lines[1:]:
        if ": " in line:
            k, v = line.split(": ", 1)
            headers[k.lower()] = v           # キーは小文字に揃えて引きやすく
    return status_line, headers, body[:200]

# --- 実験1：ふつうのページ（あるはず）→ 200 が返る ---
print("===== example.com の \"/\"（存在するページ）=====")
status, headers, body = http_get("example.com", "/")
print("ステータス行 :", status)
print("Server      :", headers.get("server", "(なし)"))
print("Content-Type:", headers.get("content-type", "(なし)"))
print("本文の先頭   :", body.strip()[:80])

# --- 実験2：存在しないページ → 404 が返る ---
print("\n===== example.com の \"/no-such-xyz\"（存在しないページ）=====")
try:
    status, headers, body = http_get("example.com", "/no-such-xyz")
    print("ステータス行 :", status, "  ← 3桁の番号に注目！")
    print("Server      :", headers.get("server", "(なし)"))
    print("Content-Type:", headers.get("content-type", "(なし)"))
except Exception as e:
    print("エラー（これも立派な“返事”）:", e)

In [ ]:
# 🌐 要ネット：Set-Cookie（合言葉を渡すヘッダ）を探す
# 大きなサイトは初回アクセス時に「合言葉(Cookie)」を渡してくることが多い。

def find_set_cookie(host, path="/"):
    status, headers, _ = http_get(host, path)
    return status, headers.get("set-cookie")

candidates = ["www.google.com", "www.yahoo.co.jp", "github.com"]
found = False
for host in candidates:
    try:
        status, cookie = find_set_cookie(host)
        print(f"----- {host} -----")
        print("ステータス行 :", status)
        if cookie:
            # Cookie は長いことがあるので先頭だけ表示
            print("Set-Cookie  :", cookie[:90] + (" …" if len(cookie) > 90 else ""))
            print("→ このサイトは『さっきの人だ』と覚えるための合言葉を渡してきた！")
            found = True
            break
        else:
            print("Set-Cookie  : (このサイトは今回 Cookie を設定していない)")
    except Exception as e:
        print(f"----- {host} ----- 接続できませんでした:", e)

if not found:
    print("\nどのサイトも今回は Set-Cookie を返しませんでした。")
    print("（タイミングや地域で変わります。Set-Cookie は『状態を覚えさせる合言葉』だと押さえればOK）")

### 返事の読み方 ── ステータスコードとヘッダ

サーバの返事は、必ず先頭に**3桁の番号（ステータスコード）**が付きます。これは故障ではなく**正式な返事**です。

- **2xx 成功**：`200 OK`＝「はい、どうぞ」。いちばん見たい番号。
- **3xx 引越し**：`301 Moved Permanently`＝「住所が変わりました、こちらへ」。自動で転送される。
- **4xx あなたのミス**：`403 Forbidden`＝「立入禁止」、`404 Not Found`＝「そのページ無いよ」。
- **5xx サーバのミス**：`500 Internal Server Error`＝「こちらの故障です、ごめんなさい」。

番号の下に続く**ヘッダ**は、荷物に貼られた**宛名シール／説明書き**です。よく見るのはこの3つ：

- **`Server`** … 返してきたソフトの名前（例：`nginx`, `gws`）。「どんな建物の受付か」。
- **`Content-Type`** … 中身の種類（例：`text/html`＝HTML、`image/png`＝画像、`application/json`＝データ）。
  ブラウザはこれを見て「文字として表示」か「画像として表示」かを決める。
- **`Set-Cookie`** … 「次回はこの**合言葉**を見せてね」という指示。これでログイン状態や買い物カゴを覚える。
  便利な反面、広告会社の Cookie は**閲覧行動の追跡**にも使われる（プライバシーの話につながる）。

> ✅ ここまでで、URLバーの裏で起きている **「GET → 200 OK → ヘッダ → 中身」** の往復を、
> 自分のコードで丸ごと観察できました。これが Web の会話＝**HTTP** の正体です。

---
## ふりかえり（提出は不要・考えてみよう）
1. 自分のグローバルIPを2進数にすると何ビットだった？ 友だちと同じだった？（→ なぜ？）
2. `www.google.com` のIPは何個返ってきた？ なぜ複数あると便利？
3. `/24` で254台。もし `/16`（上位16ビットが町名）なら何台置ける？ 計算してみよう。
4. `example.com` の `/`(存在する) と `/no-such-xyz`(存在しない) で、ステータスの番号はどう違った？ なぜ「404」は故障ではなく“正式な返事”と言える？
5. `Content-Type` が `text/html` だと、ブラウザは中身をどう扱う？ もし `image/png` ならどうなる？
6. `Set-Cookie` は何のためにある？ 便利な面と、気をつけたい面（プライバシー）をそれぞれ挙げてみよう。

> 次のノート（体験ワーク②）では、この「住所への旅」を**安全に**するしくみ（暗号・HTTPS）を体験します。